# Week 4 — New worlds, honest claims
*Albania Now — a Free Focus program, built with Chicago First. Run cells top to bottom. First: **File → Save a copy in Drive**.*

## 1. New terrain — rougher, fainter, junkier
Everything you built this month, against ground it wasn't tuned on.

In [ ]:
import numpy as np

def make_terrain(craters, size=90, seed=1, noise=15, specks=0.006):
    """A synthetic orbital image: bright plain, shadowed crater bowls,
    sunlit rims, camera noise. craters = list of (cx, cy, r)."""
    rng = np.random.default_rng(seed)
    img = 180 + rng.integers(-noise, noise, (size, size)).astype(float)
    yy, xx = np.mgrid[0:size, 0:size]
    for cx, cy, r in craters:
        d = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        bowl = d < r
        img[bowl & (xx < cx)] = 55 + rng.integers(0, 20, int((bowl & (xx < cx)).sum()))
        img[bowl & (xx >= cx)] = 225
        img[(d >= r) & (d < r + 1.5) & (xx > cx)] = 240
    sp = rng.random((size, size)) < specks
    img[sp] = 45
    return np.clip(img, 0, 255)

# the new world: more noise, more specks, and three FAINT craters whose
# shadows sit near ordinary-ground brightness — the ones a strict
# threshold loses first
TRUE_CRATERS = [(12, 20, 5), (35, 12, 7), (68, 18, 4), (25, 48, 8),
                (52, 40, 5), (80, 52, 6), (15, 78, 4), (60, 74, 9), (84, 82, 4)]
img = make_terrain(TRUE_CRATERS, seed=7, noise=22, specks=0.03)
yy, xx = np.mgrid[0:90, 0:90]
for cx, cy, r in [(68, 18, 4), (15, 78, 4), (84, 82, 4)]:   # the faint three
    d = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    img[(d < r) & (img < 110)] += 45
# boulder shadows: dark-ish patches that are NOT craters — junk with size
for bx, by in [(44, 70), (8, 44), (75, 35), (30, 28), (55, 12)]:
    img[by:by + 3, bx:bx + 5] = 108
print(img.shape, "pixels, values", int(img.min()), "to", int(img.max()))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.title("The new world — 9 true craters, if you can defend them")
plt.show()
plt.hist(img.ravel(), bins=40, color="#7E1B14"); plt.title("Start here, as always")
plt.show()

## 2. Your detector (bring week 2 forward)

In [ ]:
def find_blobs(mask):
    """Group touching True pixels into blobs (the paint-bucket trick)."""
    seen = np.zeros_like(mask, dtype=bool)
    blobs = []
    H, W = mask.shape
    for y0 in range(H):
        for x0 in range(W):
            if mask[y0, x0] and not seen[y0, x0]:
                stack, px = [(y0, x0)], []
                seen[y0, x0] = True
                while stack:
                    y, x = stack.pop()
                    px.append((y, x))
                    for dy, dx in ((1,0), (-1,0), (0,1), (0,-1)):
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < H and 0 <= nx < W and mask[ny, nx] and not seen[ny, nx]:
                            seen[ny, nx] = True
                            stack.append((ny, nx))
                blobs.append(px)
    return blobs

In [ ]:
THRESHOLD = 100     # retune for THIS terrain — the old value is a guess here
MIN_SIZE = 8
mask = img < THRESHOLD
blobs = find_blobs(mask)
dets = [b for b in blobs if len(b) >= MIN_SIZE]
print(len(dets), "detections")

## 3. Measure your errors against the labeled strip
The true crater list is above (it is the labeled data). A detection counts as a hit if its pixels reach within 2 of a true center.

In [ ]:
import numpy as np

def score(dets, truths, tol=2):
    hits = 0
    used = set()
    for i, (cx, cy, r) in enumerate(truths):
        for b in dets:
            ys = np.array([p[0] for p in b]); xs = np.array([p[1] for p in b])
            if np.min(np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)) <= r + tol:
                hits += 1
                used.add(id(b))
                break
    false_alarms = sum(1 for b in dets if id(b) not in used)
    return hits, len(truths) - hits, false_alarms

hits, misses, fa = score(dets, TRUE_CRATERS)
print(f"hits {hits} · misses {misses} · false alarms {fa}")

## 4. Walk the seesaw
Try 3–4 thresholds; record (threshold, misses, false alarms) for each; pick the one you can defend.

In [ ]:
for T in (85, 95, 105, 115):
    m = img < T
    d = [b for b in find_blobs(m) if len(b) >= MIN_SIZE]
    h, mi, f = score(d, TRUE_CRATERS)
    print(f"threshold {T}: {len(d)} detections — misses {mi}, false alarms {f}")

## 5. The build — the detection report
Write it like a professional: count, settings, miss rate, false-alarm rate, size floor, and the three-sentence finding (measured / found / not proven — remember this terrain is synthetic and the labels were free; on Titan nobody hands you TRUE_CRATERS).

**Turn-in:** notebook + screenshot of detections and report. **This completes the sprint — Dr. Nixon's lecture list awaits.**

**My report:**
- Count: …
- Settings: …
- Misses: … / False alarms: …
- Size floor: …
- Finding: 1. Measured … 2. Found … 3. Not proven …